# Strategy Comparison Backtest

Compares HMM and MS-VAR against three benchmarks over the 2018-2025 test period:
- 60/40 equity-bond portfolio
- Equal-weight equity-bond-gold portfolio
- Buy-and-hold equity portfolio

Requires pre-computed target weights from notebooks 03 and 04.

## Cost and Tax Treatment

Transaction costs: 10 bps applied to traded notional (sum of absolute weight changes).
A 1-month signal lag is applied for model strategies to prevent look-ahead bias.
Tax sensitivity is reported separately as a scenario analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.abspath(".."))

from src.constants import ASSET_COLS
from src.data_loader import load_asset_data, load_target_weights, validate_target_weights
from src.benchmarks import (build_constant_weights, build_equal_weight_benchmark,
                              build_sixty_forty_benchmark, build_equity_only_benchmark)
from src.backtest import backtest_with_transaction_costs, apply_asset_level_tax_sensitivity
from src.metrics import (annualized_return, annualized_volatility, sharpe_ratio,
                          sortino_ratio, max_drawdown, information_ratio,
                          summarize_performance)

DATA_DIR           = Path("../data")
OUTPUT_TABLES_DIR  = Path("../outputs/tables")
OUTPUT_FIGURES_DIR = Path("../outputs/figures")

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRANSACTION_COST_BPS = 10

MODEL_SIGNAL_START_DATES = {
    "HMM":   pd.Timestamp("2018-10-31"),
    "MSVAR": pd.Timestamp("2018-11-30"),
}

ASSET_TAX_RATES = {
    "No Tax": {"index_fund": 0.00, "treasury_fund": 0.00, "gold_fund": 0.00},
    "Long-Term Capital Gain Sensitivity": {"index_fund": 0.15, "treasury_fund": 0.15, "gold_fund": 0.15},
    "Gold Collectibles Sensitivity":      {"index_fund": 0.15, "treasury_fund": 0.15, "gold_fund": 0.28},
    "High Short-Term Sensitivity":        {"index_fund": 0.35, "treasury_fund": 0.35, "gold_fund": 0.35},
}

In [ ]:
asset_returns = load_asset_data(DATA_DIR / "market_clean.csv")[ASSET_COLS].sort_index()
print(asset_returns.shape)
print(f"Asset return range: {asset_returns.index.min().date()} to {asset_returns.index.max().date()}")
asset_returns.head()

In [ ]:
model_weight_files = {
    "HMM":   OUTPUT_TABLES_DIR / "hmm_target_weights.csv",
    "MSVAR": OUTPUT_TABLES_DIR / "msvar_target_weights.csv",
}

model_weights = {}
for model_name, path in model_weight_files.items():
    if path.exists():
        weights = load_target_weights(path)
        validate_target_weights(weights)
        model_weights[model_name] = weights
        print(f"Loaded {model_name} target weights from {path}")
        print(weights.head())
    else:
        print(f"Missing {model_name} target weights: {path}")

print(f"\nAvailable model weight files: {len(model_weights)}")

In [ ]:
benchmark_weights = {
    "60_40":           build_sixty_forty_benchmark(asset_returns.index),
    "Equal_Weight":    build_equal_weight_benchmark(asset_returns.index),
    "Buy_Hold_Equity": build_equity_only_benchmark(asset_returns.index),
}
list(benchmark_weights.keys())

In [ ]:
model_weights_test = {}

for model_name, weights in model_weights.items():
    signal_start = MODEL_SIGNAL_START_DATES[model_name]
    model_weights_test[model_name] = weights.loc[weights.index >= signal_start].copy()
    print(
        f"{model_name} signal test period starts at {signal_start.date()} "
        f"with {len(model_weights_test[model_name])} signal observations."
    )

all_target_weights = {}
all_target_weights.update(model_weights_test)
all_target_weights.update(benchmark_weights)

cost_results_raw = {}

for strategy_name, weights in all_target_weights.items():
    is_model_strategy = strategy_name in model_weights_test

    result = backtest_with_transaction_costs(
        target_weights=weights,
        asset_returns=asset_returns,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        apply_signal_lag=is_model_strategy,
        include_initial_trade_cost=False,
    )

    cost_results_raw[strategy_name] = result

# Align all strategies to the same realized-return comparison period
common_result_index = None
for result in cost_results_raw.values():
    if common_result_index is None:
        common_result_index = result.index
    else:
        common_result_index = common_result_index.intersection(result.index)

cost_results = {
    strategy_name: result.loc[common_result_index].copy()
    for strategy_name, result in cost_results_raw.items()
}

for strategy_name, result in cost_results.items():
    first_date = result.index[0]
    result.loc[first_date, "trade_notional"] = 0.0
    result.loc[first_date, "one_way_turnover"] = 0.0
    result.loc[first_date, "transaction_cost"] = 0.0
    result.loc[first_date, "net_return_after_costs"] = result.loc[first_date, "gross_return"]

for strategy_name, result in cost_results.items():
    output_path = OUTPUT_TABLES_DIR / f"{strategy_name.lower()}_monthly_returns_with_costs.csv"
    result.to_csv(output_path)
    print(f"Saved {strategy_name}: {result.index.min().date()} to {result.index.max().date()}, {len(result)} months")

test_period_info = pd.DataFrame({
    "Strategy": list(cost_results.keys()),
    "Signal Start Date": [
        MODEL_SIGNAL_START_DATES.get(s, pd.NaT) for s in cost_results.keys()
    ],
    "Realized Return Start Date": [
        cost_results[s].index.min() for s in cost_results.keys()
    ],
    "Realized Return End Date": [
        cost_results[s].index.max() for s in cost_results.keys()
    ],
    "Number of Return Observations": [
        len(cost_results[s]) for s in cost_results.keys()
    ],
})
test_period_info.to_csv(OUTPUT_TABLES_DIR / "test_period_info.csv", index=False)
test_period_info

## Performance Metrics

Primary comparison uses net returns after transaction costs.

In [ ]:
benchmark_reference_net = cost_results["60_40"]["net_return_after_costs"]

net_strategy_comparison = pd.DataFrame({
    strategy_name: summarize_performance(
        result["net_return_after_costs"],
        benchmark_returns=None if strategy_name == "60_40" else benchmark_reference_net,
    )
    for strategy_name, result in cost_results.items()
})

metric_order = [
    "Annualized Return",
    "Annualized Volatility",
    "Sharpe Ratio",
    "Sortino Ratio",
    "Max Drawdown",
    "Information Ratio vs 60/40",
]

net_strategy_comparison = net_strategy_comparison.reindex(
    [m for m in metric_order if m in net_strategy_comparison.index]
)

net_strategy_comparison.to_csv(OUTPUT_TABLES_DIR / "net_after_cost_strategy_comparison.csv")
net_strategy_comparison

In [ ]:
net_return_series = pd.DataFrame({
    strategy_name: result["net_return_after_costs"]
    for strategy_name, result in cost_results.items()
})

net_cumulative_returns = (1 + net_return_series).cumprod() - 1
net_cumulative_returns.to_csv(OUTPUT_TABLES_DIR / "net_after_cost_cumulative_returns.csv")

plt.figure(figsize=(12, 6))
for col in net_cumulative_returns.columns:
    plt.plot(net_cumulative_returns.index, net_cumulative_returns[col], label=col)
plt.title("Strategy Comparison: Net Performance After Transaction Costs")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "net_after_cost_strategy_comparison.png", dpi=300)
plt.show()

In [ ]:
ranking_metrics = [
    "Annualized Return",
    "Sharpe Ratio",
    "Sortino Ratio",
    "Max Drawdown",
    "Information Ratio vs 60/40",
]

strategy_ranking = net_strategy_comparison.loc[
    [m for m in ranking_metrics if m in net_strategy_comparison.index]
].T

strategy_ranking["Return Rank"]  = strategy_ranking["Annualized Return"].rank(ascending=False)
strategy_ranking["Sharpe Rank"]  = strategy_ranking["Sharpe Ratio"].rank(ascending=False)
strategy_ranking["Sortino Rank"] = strategy_ranking["Sortino Ratio"].rank(ascending=False)
strategy_ranking["Drawdown Rank"] = strategy_ranking["Max Drawdown"].rank(ascending=False)
strategy_ranking = strategy_ranking.sort_values("Sharpe Rank")

strategy_ranking.to_csv(OUTPUT_TABLES_DIR / "net_after_cost_strategy_ranking.csv")
strategy_ranking

## Implementation Cost Summary

In [ ]:
implementation_cost_summary = pd.DataFrame({
    strategy_name: pd.Series({
        "Average Trade Notional": result["trade_notional"].mean(),
        "Average One-Way Turnover": result["one_way_turnover"].mean(),
        "Average Monthly Transaction Cost": result["transaction_cost"].mean(),
    })
    for strategy_name, result in cost_results.items()
})
implementation_cost_summary.to_csv(OUTPUT_TABLES_DIR / "implementation_cost_summary.csv")
implementation_cost_summary

In [ ]:
for strategy_name, result in cost_results.items():
    cumulative = (1 + result[["gross_return", "net_return_after_costs"]]).cumprod() - 1

    plt.figure(figsize=(12, 6))
    plt.plot(cumulative.index, cumulative["gross_return"], label=f"{strategy_name} Gross")
    plt.plot(cumulative.index, cumulative["net_return_after_costs"], label=f"{strategy_name} Net")
    plt.title(f"{strategy_name}: Gross vs Net Performance After Transaction Costs")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    output_path = OUTPUT_FIGURES_DIR / f"{strategy_name.lower()}_gross_vs_net_transaction_costs.png"
    plt.savefig(output_path, dpi=300)
    plt.show()

## Tax Sensitivity

Estimates potential tax drag from rebalancing sales under different rate assumptions.
This is a sensitivity scenario, not a full tax-lot accounting model.

In [ ]:
tax_results = {}

for strategy_name, weights in all_target_weights.items():
    strategy_tax_results = {}
    is_model_strategy = strategy_name in model_weights_test

    for scenario_name, tax_rates in ASSET_TAX_RATES.items():
        tax_drag = apply_asset_level_tax_sensitivity(
            target_weights=weights,
            asset_returns=asset_returns,
            tax_rates=tax_rates,
            apply_signal_lag=is_model_strategy,
        )

        base_returns = cost_results[strategy_name]["net_return_after_costs"]
        common_index = base_returns.index.intersection(tax_drag.index)

        tax_drag_aligned = tax_drag.loc[common_index, "tax_drag"].copy()
        tax_drag_aligned.iloc[0] = 0.0

        after_tax_returns = base_returns.loc[common_index] - tax_drag_aligned
        strategy_tax_results[scenario_name] = after_tax_returns

    tax_results[strategy_name] = pd.DataFrame(strategy_tax_results)
    output_path = OUTPUT_TABLES_DIR / f"{strategy_name.lower()}_tax_sensitivity_returns.csv"
    tax_results[strategy_name].to_csv(output_path)
    print(f"Saved {strategy_name} tax sensitivity returns.")

In [ ]:
tax_summary_blocks = []

for strategy_name, returns_df in tax_results.items():
    scenario_summaries = {}

    for scenario in returns_df.columns:
        scenario_benchmark = None if strategy_name == "60_40" else tax_results["60_40"][scenario]
        scenario_summaries[scenario] = summarize_performance(
            returns_df[scenario],
            benchmark_returns=scenario_benchmark,
        )

    tax_summary = pd.DataFrame(scenario_summaries)
    tax_summary.columns = [f"{strategy_name} - {col}" for col in tax_summary.columns]
    tax_summary_blocks.append(tax_summary)

tax_sensitivity_summary = pd.concat(tax_summary_blocks, axis=1)
tax_sensitivity_summary.to_csv(OUTPUT_TABLES_DIR / "tax_sensitivity_summary.csv")
tax_sensitivity_summary

In [ ]:
for strategy_name, returns_df in tax_results.items():
    cumulative = (1 + returns_df).cumprod() - 1

    plt.figure(figsize=(12, 6))
    for col in cumulative.columns:
        plt.plot(cumulative.index, cumulative[col], label=col)
    plt.title(f"{strategy_name}: Tax Sensitivity Analysis")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    output_path = OUTPUT_FIGURES_DIR / f"{strategy_name.lower()}_tax_sensitivity.png"
    plt.savefig(output_path, dpi=300)
    plt.show()